# Verified measure-and-conquer running-time analysis

This notebook analyzes the `Search` algorithm for **$\ell$-Forest Deletion Set** and the outer enumeration in `ComputingProductList`, as specified in `4_lforest_deletion.tex` of the accompanying paper. It computes the two bases in $O^*(K^{\hat k}L^\ell)$, verifies the original $(17,64)$ bound, and gives **exact** optimizers for $K$ and $L$. The $K$-only optimum is chosen to minimize $L$ among all choices attaining the smallest $K$.

## Branching vectors and initial measure

With $0<\alpha<\beta$, the branching vectors in the paper are
$$
(1,1,1,\min\{1,\alpha\}) \quad\text{and}\quad(1,\beta-\alpha).
$$
Let $p,q>1$ denote their respective branching numbers, defined by
$$
\frac{3}{p}+p^{-\min\{1,\alpha\}}=1,\qquad
\frac1q+q^{-(\beta-\alpha)}=1.
$$
Set $c=\max\{p,q\}$. On input $S\subseteq\hat S$, the paper bounds the initial measure by
$$
\mu\le(1+\alpha)(\hat k-|S|)+(1+\beta)\ell.
$$
Summing the corresponding search-tree bound over $S\subseteq\hat S$ gives
$$
\sum_{S\subseteq\hat S}c^{(1+\alpha)(\hat k-|S|)+(1+\beta)\ell}
\le (1+c^{1+\alpha})^{\hat k}c^{(1+\beta)\ell}.
$$
Hence the **complete** running-time bound is
$$
O^*(K^{\hat k}L^\ell),\qquad
K=1+c^{1+\alpha},\quad L=c^{1+\beta}.
$$
The factor $1+$ in $K$ is required by the outer enumeration.

## Exact optimization (no heuristic assumption)

For $0<\alpha\le1$, the standard branching equation implies
$$
p^{-\alpha}=1-\frac3p,\quad
p^{1+\alpha}=\frac{p^2}{p-3},\quad p\ge4.
$$
For fixed $\alpha$, taking $q=p$ simultaneously minimizes $L$ and does not increase $K$: if $q>p$, increasing $\beta$ until $q=p$ decreases both $K$ and $L$; if $q<p$, decreasing $\beta$ until $q=p$ leaves $K$ unchanged and decreases $L$. (In the first case, $L=q^{\alpha+2}/(q-1)$ is increasing in $q\ge p\ge4$.) The choices $\alpha>1$ cannot improve either optimum. Thus we may parameterize the optimal choices by $p\ge4$, with
$$
\alpha(p)=\log_p\frac{p}{p-3},\qquad
\beta(p)=\log_p\frac{p^2}{(p-3)(p-1)},
$$
$$
K(p)=1+\frac{p^2}{p-3},\qquad
L(p)=\frac{p^3}{(p-3)(p-1)}.
$$

**Minimizing $K$.** The identity
$$
K(p)-13=\frac{(p-6)^2}{p-3}\ge0
$$
shows that $K\ge13$, with equality at $p=6$. Among all choices with $K=13$, the smallest $L$ occurs at $q=p=6$, giving
$$
\boxed{\alpha=\log_6 2,\quad\beta=\log_6(12/5),\quad K=13,\quad L=72/5=14.4.}
$$

**Minimizing $L$.** Differentiating $L(p)$ shows that its unique minimum on $p\ge4$ is at $p=4+\sqrt7$. Therefore,
$$
\boxed{L_{\min}=5+\frac{7\sqrt7}{2}\approx14.260129589,\qquad
K=\frac{13+5\sqrt7}{2}\approx13.114378278.}
$$
Since $\max\{K,L\}\ge L\ge L_{\min}$ and $K<L$ at this optimizer, it also minimizes the **common base** $\max\{K,L\}$.

For fixed $k,\ell>0$, minimizing $K^kL^\ell$ reduces to a one-dimensional minimization of $k\log K(p)+\ell\log L(p)$ over $6\le p\le4+\sqrt7$.

## Implementation and outputs

`optimize('k')` returns the exact $K$-minimizing parameters, using $L$ as a tie-breaker. `optimize('ell')` and `optimize('common')` return the other exact optimizer. `optimize('instance', k=..., ell=...)` optimizes for a given pair of parameters. Optional nonstandard parameter bounds are supported through a numerical fallback, which is not a certificate of global optimality.

All displayed decimal results are approximations; the exact expressions above establish the bounds without numerical optimization.

In [ ]:
"""Verified measure-and-conquer analysis for ell-Forest Deletion Set.

The formula includes both Search and the outer enumeration S subseteq hat(S).
For the unrestricted domain 0 < alpha < beta, the two extreme bases admit
exact optimizers.  For K, ties are broken by minimizing L.
"""

from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Sequence

from scipy.optimize import brentq, differential_evolution, minimize_scalar


@dataclass(frozen=True)
class Estimate:
    alpha: float
    beta: float
    standard_root: float
    cycle_root: float
    worst_root: float
    k_base: float
    ell_base: float
    common_base: float


def branching_number(decreases: Sequence[float]) -> float:
    """Unique root r >= 1 of sum(r**(-d) for d in decreases) = 1."""
    decreases = tuple(float(d) for d in decreases)
    if not decreases or any(not math.isfinite(d) or d <= 0 for d in decreases):
        raise ValueError("Every branching decrease must be positive and finite.")
    if len(decreases) == 1:
        return 1.0

    def characteristic(root: float) -> float:
        return sum(root ** (-d) for d in decreases) - 1.0

    hi = 2.0
    while characteristic(hi) > 0:
        hi *= 2.0
        if not math.isfinite(hi):
            raise OverflowError("Could not bracket the branching root.")
    return brentq(characteristic, 1.0, hi, xtol=1e-13, rtol=1e-13)


def roots(alpha: float, beta: float) -> tuple[float, float]:
    """Branching numbers of the standard and cycle branching vectors."""
    if not (math.isfinite(alpha) and math.isfinite(beta) and 0 < alpha < beta):
        raise ValueError("Require finite parameters 0 < alpha < beta.")
    standard = branching_number((1, 1, 1, min(1.0, alpha)))
    cycle = branching_number((1, beta - alpha))
    return standard, cycle


def evaluate(alpha: float, beta: float) -> Estimate:
    """Bases in O*(K**hat_k * L**ell), including outer enumeration."""
    p, q = roots(alpha, beta)
    c = max(p, q)
    K = 1.0 + c ** (1.0 + alpha)
    L = c ** (1.0 + beta)
    return Estimate(alpha, beta, p, q, c, K, L, max(K, L))


def parameters_for_equal_roots(p: float) -> tuple[float, float]:
    """For p >= 4, obtain alpha <= 1 and beta with both branching roots p."""
    if not (math.isfinite(p) and p >= 4.0):
        raise ValueError("Require p >= 4.")
    log_p = math.log(p)
    alpha = math.log(p / (p - 3.0)) / log_p
    beta = alpha + math.log(p / (p - 1.0)) / log_p
    return alpha, beta


def optimal_k() -> Estimate:
    """Globally minimize K; among all K-minimizers, minimize L.

    At p=6, K=13 and L=72/5=14.4 exactly.
    """
    return evaluate(math.log(2.0) / math.log(6.0),
                    math.log(12.0 / 5.0) / math.log(6.0))


def optimal_ell() -> Estimate:
    """Globally minimize L, attained when p=q=4+sqrt(7)."""
    return evaluate(*parameters_for_equal_roots(4.0 + math.sqrt(7.0)))


def optimal_instance(k: int, ell: int) -> Estimate:
    """Minimize K**k L**ell by a deterministic one-dimensional search.

    For k,ell>0, an optimum lies on p=q with 6 <= p <= 4+sqrt(7).
    """
    if not isinstance(k, int) or not isinstance(ell, int) or k < 0 or ell < 0:
        raise ValueError("k and ell must be nonnegative integers.")
    if k == 0:
        return optimal_ell()
    if ell == 0:
        return optimal_k()

    def log_bound(p: float) -> float:
        K = 1.0 + p*p / (p - 3.0)
        L = p**3 / ((p - 3.0)*(p - 1.0))
        return k*math.log(K) + ell*math.log(L)

    res = minimize_scalar(log_bound, bounds=(6.0, 4.0 + math.sqrt(7.0)),
                          method='bounded', options={'xatol': 1e-14})
    if not res.success:
        raise RuntimeError(res.message)
    return evaluate(*parameters_for_equal_roots(float(res.x)))


def _in_bounds(est: Estimate, alpha_bounds, beta_bounds) -> bool:
    return (alpha_bounds[0] <= est.alpha <= alpha_bounds[1]
            and beta_bounds[0] <= est.beta <= beta_bounds[1])


def optimize(
    objective: str = 'common',
    alpha_bounds: tuple[float, float] = (0.05, 1.5),
    beta_bounds: tuple[float, float] = (0.06, 3.0),
    k: int | None = None,
    ell: int | None = None,
    seed: int = 0,
) -> Estimate:
    """Minimize the complete bound over user-specified parameter ranges.

    The default ranges contain exact unrestricted optimizers for 'k', 'ell'
    and 'common'; the 'k' result additionally minimizes L among K-minimizers.
    'instance' performs a deterministic 1D minimization when admissible.
    For unusual bounds excluding an optimizer, fall back to 2D numerical
    differential evolution (which does not certify global optimality).
    """
    if objective not in {'common', 'k', 'ell', 'instance'}:
        raise ValueError("Unknown objective.")
    if (len(alpha_bounds) != 2 or len(beta_bounds) != 2
        or not (0 < alpha_bounds[0] < alpha_bounds[1]
                and 0 < beta_bounds[0] < beta_bounds[1])):
        raise ValueError("Bounds must be positive increasing intervals.")
    if objective == 'instance' and (k is None or ell is None):
        raise ValueError("'instance' requires k and ell.")

    if objective == 'k':
        candidate = optimal_k()
    elif objective == 'ell' or objective == 'common':
        candidate = optimal_ell()
    else:
        candidate = optimal_instance(k, ell)
    if _in_bounds(candidate, alpha_bounds, beta_bounds):
        return candidate

    # Only required for restricted parameter boxes excluding the solution.
    def log_objective(x: Sequence[float]) -> float:
        alpha, beta = map(float, x)
        if alpha <= 0 or beta <= alpha:
            return 1e8 + 1e6 * max(0, alpha - beta)
        est = evaluate(alpha, beta)
        log_k, log_ell = math.log(est.k_base), math.log(est.ell_base)
        if objective == 'common':
            return max(log_k, log_ell)
        if objective == 'k':
            return log_k
        if objective == 'ell':
            return log_ell
        return k*log_k + ell*log_ell

    res = differential_evolution(log_objective,
                                 bounds=(alpha_bounds, beta_bounds),
                                 seed=seed, tol=1e-10, polish=True)
    if not res.success:
        raise RuntimeError(res.message)
    return evaluate(*map(float, res.x))


def residuals(est: Estimate) -> tuple[float, float]:
    return (3.0 / est.standard_root
            + est.standard_root ** (-min(1.0, est.alpha)) - 1.0,
            1.0 / est.cycle_root
            + est.cycle_root ** (-(est.beta - est.alpha)) - 1.0)


def print_estimate(title: str, est: Estimate) -> None:
    rp, rq = residuals(est)
    print(title)
    print(f'  alpha = {est.alpha:.12f}')
    print(f'  beta  = {est.beta:.12f}')
    print(f'  standard root p = {est.standard_root:.12f}')
    print(f'  cycle root q    = {est.cycle_root:.12f}')
    print(f'  worst root c    = {est.worst_root:.12f}')
    print(f'  K = {est.k_base:.12f}')
    print(f'  L = {est.ell_base:.12f}')
    print(f'  max(K,L) = {est.common_base:.12f}')
    print(f'  characteristic residuals = ({rp:.3e}, {rq:.3e})')




In [ ]:
print_estimate('Original paper', evaluate(1.0, 2.0))
print()
print_estimate('Minimize K (and then L)', optimize('k'))
print()
print_estimate('Minimize L', optimize('ell'))
print()
print_estimate('Minimize max(K,L)', optimize('common'))
print()
print_estimate('Instance k=10, ell=4', optimize('instance', k=10, ell=4))

Original paper
  alpha = 1.000000000000
  beta  = 2.000000000000
  standard root p = 4.000000000000
  cycle root q    = 2.000000000000
  worst root c    = 4.000000000000
  K = 17.000000000000
  L = 64.000000000000
  max(K,L) = 64.000000000000
  characteristic residuals = (0.000e+00, 0.000e+00)

Minimize K (and then L)
  alpha = 0.386852807235
  beta  = 0.488608405531
  standard root p = 6.000000000000
  cycle root q    = 6.000000000000
  worst root c    = 6.000000000000
  K = 13.000000000000
  L = 14.400000000000
  max(K,L) = 14.400000000000
  characteristic residuals = (-2.220e-16, 0.000e+00)

Minimize L
  alpha = 0.317012850177
  beta  = 0.403114425534
  standard root p = 6.645751311065
  cycle root q    = 6.645751311065
  worst root c    = 6.645751311065
  K = 13.114378277661
  L = 14.260129588726
  max(K,L) = 14.260129588726
  characteristic residuals = (0.000e+00, 0.000e+00)

Minimize max(K,L)
  alpha = 0.317012850177
  beta  = 0.403114425534
  standard root p = 6.645751311065
  c